# Chatbot Beca 18 — RAG Pipeline
**Documento fuente:** Resolución Directoral Ejecutiva N.° 033-2026-MINEDU/VMGI-PRONABEC  
**Modelos:** `gemini-embedding-001` (768 dim) · `gemini-2.5-flash`  
**Vector DB:** ChromaDB (distancia coseno)

```
PDF → extracción → chunking → embeddings → ChromaDB
    → pregunta → query embedding → top-k retrieval
    → LLM con contexto → respuesta citada
```

---
## Paso 0 — Configuración del entorno

In [1]:
# Instalación de dependencias
import subprocess, sys
pkgs = [
    'pypdf', 'tiktoken', 'langchain-text-splitters',
    'google-genai', 'chromadb', 'ipywidgets', 'tqdm', 'python-dotenv'
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--break-system-packages', '-q'] + pkgs,
    capture_output=True
)
print('Dependencias instaladas ✓')

Dependencias instaladas ✓


In [2]:
import importlib.metadata, os
from dotenv import load_dotenv

# Cargar API key desde .env (nunca hardcodeada)
# Buscar .env en el directorio padre del notebook
env_path = os.path.join(os.path.dirname(os.path.abspath('.')), '.env')
load_dotenv(env_path)
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')

if not GEMINI_API_KEY:
    raise EnvironmentError(
        'GEMINI_API_KEY no encontrada.\n'
        'Crea un archivo .env en la raíz del repositorio con:\n'
        '    GEMINI_API_KEY=tu_clave_aqui\n'
        'Obtén tu clave en: https://aistudio.google.com/app/apikey'
    )
else:
    print(f'API Key cargada ✓  (termina en ...{GEMINI_API_KEY[-4:]})')

# Versiones de paquetes
for pkg in ['pypdf', 'tiktoken', 'langchain-text-splitters',
            'google-genai', 'chromadb', 'ipywidgets', 'tqdm', 'python-dotenv']:
    try:
        v = importlib.metadata.version(pkg)
        print(f'  {pkg:<30} {v}')
    except Exception:
        print(f'  {pkg:<30} (versión no disponible)')

API Key cargada ✓  (termina en ...tKAg)
  pypdf                          6.11.0
  tiktoken                       0.13.0
  langchain-text-splitters       1.1.2
  google-genai                   2.3.0
  chromadb                       1.5.9
  ipywidgets                     8.1.8
  tqdm                           4.67.3
  python-dotenv                  1.2.2


---
## Paso 1 — Extracción de texto PDF

In [5]:
from pathlib import Path

PDF_PATH = Path("../data/beca18_reglamento.pdf")

print("Existe:", PDF_PATH.exists())
print("Tamaño MB:", round(PDF_PATH.stat().st_size / (1024 * 1024), 2))

with open(PDF_PATH, "rb") as f:
    print("Primeros bytes:", f.read(5))

Existe: True
Tamaño MB: 3.82
Primeros bytes: b'%PDF-'


In [6]:
import re
from pypdf import PdfReader

# Ruta al PDF (relativa al notebook en notebooks/)
# El notebook vive en notebooks/, los datos en data/
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('beca18_rag_chatbot.ipynb'))
BASE_DIR = os.path.dirname(NOTEBOOK_DIR)
PDF_PATH = os.path.join(BASE_DIR, 'data', 'beca18_reglamento.pdf')

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extrae texto página a página con marcador [PAGE N] y limpieza ligera."""
    reader = PdfReader(pdf_path)
    pages_text = []

    for i, page in enumerate(reader.pages, start=1):
        raw = page.extract_text() or ''
        # Limpieza ligera
        text = re.sub(r'[ \t]+', ' ', raw)          # espacios múltiples
        text = re.sub(r'\n{3,}', '\n\n', text)      # saltos de línea excesivos
        text = re.sub(r'(?<!\.)\n(?!\n)', ' ', text) # saltos aislados → espacio
        text = text.strip()
        pages_text.append(f'[PAGE {i}]\n{text}')

    return '\n\n'.join(pages_text)


full_text = extract_text_from_pdf(PDF_PATH)

total_chars = len(full_text)
total_words = len(full_text.split())
print(f'Páginas extraídas : {full_text.count("[PAGE ")} páginas')
print(f'Total caracteres  : {total_chars:,}')
print(f'Total palabras    : {total_words:,}')
print(f'\nMuestra (primeros 500 caracteres):\n{full_text[:500]}')

Páginas extraídas : 138 páginas
Total caracteres  : 374,509
Total palabras    : 55,202

Muestra (primeros 500 caracteres):
[PAGE 1]
Resolución Directoral Ejecutiva  Nº 033-2026-MINEDU/VMGI-PRONABEC     Lima, 24 de febrero de 2026    VISTOS:    El Informe N° 451-2026-MINEDU/VMGI-PRONABEC-DIBEC-SES, suscrito por  la Dirección de Gestión de Becas y la Dirección de Acompañamiento Socioemocional y  Bienestar; el Informe N° 042-2026-MINEDU/VMGI-PRONABEC-OPP de la Oficina de  Planeamiento y Presupuesto; el Informe N ° 048-2026-MINEDU/VMGI-PRONABEC-OAJ  de la Oficina de Asesoría Jurídica, y;    CONSIDERANDO:    Que, la Ley 


---
## Paso 2 — Justificación de tokenización y segmentación

In [18]:
# tiktoken requiere descarga de vocabulario — en algunos entornos sin internet
# usamos una aproximación (1 token ≈ 4 caracteres en español)
try:
    import tiktoken
    enc = tiktoken.get_encoding('cl100k_base')
    tokens = enc.encode(full_text)
    total_tokens = len(tokens)
    method = 'tiktoken cl100k_base'
except Exception:
    # Fallback: aproximación estándar para español
    total_tokens = len(full_text) // 4
    method = 'aproximación (chars/4) — tiktoken no disponible sin red'

n_pages = full_text.count('[PAGE ')
print(f'Codificación      : {method}')
print(f'Total de tokens   : {total_tokens:,}')
print(f'Tokens por página ≈ {total_tokens // n_pages:,}')

Codificación      : tiktoken cl100k_base
Total de tokens   : 108,737
Tokens por página ≈ 787


### Justificación del tamaño de chunk

El documento contiene aproximadamente **108,737 tokens**, por lo que no puede procesarse como un único bloque de texto para recuperación semántica. Por ello, se utiliza una estrategia de segmentación con `chunk_size = 400` tokens y `chunk_overlap = 60` tokens.

El modelo `gemini-embedding-001` admite entradas de hasta **8,192 tokens** por solicitud de incrustación. En ese sentido, un tamaño de 400 tokens representa cerca del **5% del límite máximo**, lo que ofrece un margen amplio de seguridad frente a posibles diferencias entre el tokenizador usado para el conteo (`cl100k_base`) y el procesamiento interno del modelo de embeddings.

Además, fragmentos de 400 tokens permiten conservar suficiente contexto normativo para responder preguntas específicas sobre requisitos, obligaciones, beneficios o condiciones de pérdida de la beca, sin generar bloques demasiado extensos que dificulten la recuperación precisa de información.

Finalmente, el solapamiento de 60 tokens, equivalente a aproximadamente el **15% del tamaño del fragmento**, reduce el riesgo de cortar ideas, artículos o condiciones normativas entre dos fragmentos consecutivos. Esto mejora la probabilidad de que el sistema recupere información completa y coherente durante la búsqueda semántica.

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

# Splitter basado en TOKENS usando tiktoken cl100k_base
# from_tiktoken_encoder mide chunk_size y chunk_overlap en tokens, no en caracteres
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name = 'cl100k_base',
    chunk_size    = 400,       # 400 tokens por chunk
    chunk_overlap = 60,        # 60 tokens de solapamiento
    separators    = ['\n\n', '\n', '. ', ' '],
)

raw_chunks = splitter.split_text(full_text)

# Función para extraer el número de página del marcador [PAGE N]
def get_page_number(chunk_text: str) -> str:
    """Extrae el número de página más reciente del marcador [PAGE N]."""
    matches = re.findall(r'\[PAGE (\d+)\]', chunk_text)
    return matches[-1] if matches else 'desconocida'

# Tokenizador para medir longitud en tokens de cada chunk
enc = tiktoken.get_encoding('cl100k_base')

# Construir lista de chunks con metadatos
chunks = []
for i, text in enumerate(raw_chunks):
    page = get_page_number(text)
    chunks.append({
        'id'  : f'chunk_{i:04d}',
        'text': text,
        'metadata': {
            'document': 'RDE-033-2026-MINEDU-VMGI-PRONABEC',
            'topic'   : 'Beca 18 - Bases del Concurso 2026',
            'language': 'es',
            'page'    : page,
            'chunk_id': i,
        }
    })

avg_chars  = sum(len(c['text']) for c in chunks) / len(chunks)
avg_tokens = sum(len(enc.encode(c['text'])) for c in chunks) / len(chunks)

print(f'Total de chunks          : {len(chunks):,}')
print(f'Longitud promedio        : {avg_chars:.0f} caracteres')
print(f'Longitud promedio        : {avg_tokens:.0f} tokens')
print(f'\nEjemplo chunk #0:')
print(f'  Tokens : {len(enc.encode(chunks[0]["text"]))}')
print(f'  Texto  : {chunks[0]["text"][:300]}...')
print(f'  Metadata: {chunks[0]["metadata"]}')

Total de chunks          : 618
Longitud promedio        : 634 caracteres
Longitud promedio        : 184 tokens

Ejemplo chunk #0:
  Tokens : 5
  Texto  : [PAGE 1]...
  Metadata: {'document': 'RDE-033-2026-MINEDU-VMGI-PRONABEC', 'topic': 'Beca 18 - Bases del Concurso 2026', 'language': 'es', 'page': '1', 'chunk_id': 0}


---
## Paso 3 — Embeddings con Gemini

In [21]:
import time
import google.genai as genai
from google.genai import types as genai_types

# Inicializar cliente
client = genai.Client(api_key=GEMINI_API_KEY)

EMBED_MODEL = 'gemini-embedding-001'


def _parse_retry_delay(error_str: str, fallback: float) -> float:
    """Extrae el retryDelay sugerido por la API del mensaje de error (en segundos)."""
    import re as _re
    # El error incluye 'retryDelay': '32s' o 'Please retry in 32.95...s'
    m = _re.search(r'retryDelay[^\d]*(\d+\.?\d*)', str(error_str))
    if m:
        return float(m.group(1)) + 3  # +3s de margen
    m2 = _re.search(r'retry in (\d+\.?\d*)s', str(error_str))
    if m2:
        return float(m2.group(1)) + 3
    return fallback


def embed_with_backoff(texts: list[str], task_type: str,
                       max_retries: int = 8) -> list[list[float]]:
    """
    Genera embeddings en lote con reintentos que respetan el retryDelay
    indicado por la API (tier gratuito: 100 req/min, ~60s de ventana).
    Lotes de 5 textos para minimizar la tasa de solicitudes.
    """
    all_embeddings = []
    batch_size = 5  # Conservador para tier gratuito

    for start in range(0, len(texts), batch_size):
        batch = texts[start: start + batch_size]
        for attempt in range(max_retries):
            try:
                response = client.models.embed_content(
                    model   = EMBED_MODEL,
                    contents= batch,
                    config  = genai_types.EmbedContentConfig(task_type=task_type)
                )
                batch_vectors = [e.values for e in response.embeddings]
                all_embeddings.extend(batch_vectors)
                break
            except Exception as e:
                err_str = str(e)
                if '429' in err_str or 'RESOURCE_EXHAUSTED' in err_str:
                    # Parsear el delay sugerido por la API y esperarlo
                    wait = _parse_retry_delay(err_str, fallback=65.0)
                else:
                    # Otro error: backoff exponencial corto
                    wait = 2 ** attempt
                print(f'    [lote {start//batch_size+1}] Reintento {attempt+1}/{max_retries} '
                      f'esperando {wait:.0f}s — {err_str[:80]}')
                time.sleep(wait)
        else:
            raise RuntimeError(
                f'Lote {start}-{start+batch_size} falló tras {max_retries} reintentos. '
                f'Considera ejecutar el notebook mañana si se agotó la cuota diaria.'
            )

        # Pausa fija entre lotes para no saturar la ventana de 1 minuto
        if start + batch_size < len(texts):
            time.sleep(0.7)  # ~85 lotes/min con batch=5 → dentro del límite de 100 req/min

    return all_embeddings


def embed_documents(texts: list[str]) -> list[list[float]]:
    """Incrusta textos para indexación (RETRIEVAL_DOCUMENT)."""
    return embed_with_backoff(texts, task_type='RETRIEVAL_DOCUMENT')


def embed_query(text: str) -> list[float]:
    """Incrusta una consulta para búsqueda (RETRIEVAL_QUERY)."""
    return embed_with_backoff([text], task_type='RETRIEVAL_QUERY')[0]


# Verificar con un texto de prueba
test_vec = embed_query('¿Cuáles son los requisitos de Beca 18?')
print(f'Dimensiones del embedding : {len(test_vec)}')
print(f'Primeros 5 valores        : {test_vec[:5]}')

    [lote 1] Reintento 1/8 esperando 1s — [Errno 11001] getaddrinfo failed
    [lote 1] Reintento 2/8 esperando 2s — [Errno 11001] getaddrinfo failed
    [lote 1] Reintento 3/8 esperando 4s — [Errno 11001] getaddrinfo failed
Dimensiones del embedding : 3072
Primeros 5 valores        : [-0.013971599, -0.0010298089, 0.033623558, -0.055446096, 0.010343513]


---
## Paso 4 — Base de datos vectorial ChromaDB

In [22]:
import chromadb
from tqdm.auto import tqdm

CHROMA_PATH     = os.path.join(BASE_DIR, 'chroma_db_beca18')
COLLECTION_NAME = 'beca18_docs'
BATCH           = 5   # Conservador para tier gratuito de Gemini

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Colección con distancia coseno
collection = chroma_client.get_or_create_collection(
    name     = COLLECTION_NAME,
    metadata = {'hnsw:space': 'cosine'}
)

total_chunks   = len(chunks)
existing_count = collection.count()
print(f'Chunks totales            : {total_chunks}')
print(f'Documentos en ChromaDB   : {existing_count}')

if existing_count >= total_chunks:
    print('✓ Colección completamente indexada — nada que hacer.')

else:
    # Identificar IDs ya indexados para reanudar sin repetir
    if existing_count > 0:
        stored = collection.get(include=[])  # solo IDs, sin datos
        indexed_ids = set(stored['ids'])
        pending = [c for c in chunks if c['id'] not in indexed_ids]
        print(f'Reanudando: {existing_count} ya indexados, '
              f'{len(pending)} pendientes.')
    else:
        pending = chunks
        print(f'Indexando {len(pending)} chunks desde cero...')

    # Indexación incremental: embed + insert por lote (no acumular todo en RAM)
    for start in tqdm(range(0, len(pending), BATCH), desc='Indexando'):
        batch_chunks = pending[start: start + BATCH]
        batch_ids    = [c['id']       for c in batch_chunks]
        batch_texts  = [c['text']     for c in batch_chunks]
        batch_metas  = [c['metadata'] for c in batch_chunks]

        # Generar embeddings para este lote (con backoff interno)
        batch_vecs = embed_documents(batch_texts)

        # Insertar inmediatamente en ChromaDB — si falla aquí no se pierde
        # lo ya insertado en lotes anteriores
        collection.add(
            ids        = batch_ids,
            documents  = batch_texts,
            embeddings = batch_vecs,
            metadatas  = batch_metas
        )

        # Pausa entre lotes para no saturar la ventana de 1 minuto
        if start + BATCH < len(pending):
            time.sleep(0.7)

    print(f'✓ Indexación completada')

print(f'\nTotal documentos en ChromaDB: {collection.count():,}')

Chunks totales            : 618
Documentos en ChromaDB   : 0
Indexando 618 chunks desde cero...


Indexando:   0%|          | 0/124 [00:00<?, ?it/s]

    [lote 1] Reintento 1/8 esperando 23s — 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu
    [lote 1] Reintento 1/8 esperando 18s — 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu
    [lote 1] Reintento 1/8 esperando 23s — 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu
    [lote 1] Reintento 1/8 esperando 22s — 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu
    [lote 1] Reintento 1/8 esperando 19s — 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu
✓ Indexación completada

Total documentos en ChromaDB: 618


---
## Paso 5 — Búsqueda semántica

In [23]:
def semantic_search(question: str, k: int = 5) -> list[dict]:
    """
    Busca los k fragmentos más relevantes para la pregunta.
    Retorna lista de dicts con: text, metadata, distance.
    """
    q_vec = embed_query(question)

    results = collection.query(
        query_embeddings = [q_vec],
        n_results        = k,
        include          = ['documents', 'metadatas', 'distances']
    )

    output = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        output.append({'text': doc, 'metadata': meta, 'distance': dist})

    return output


# Prueba con pregunta de ejemplo
test_question = '¿Cuáles son los requisitos socioeconómicos para postular a Beca 18?'
test_results  = semantic_search(test_question, k=3)

print(f'Pregunta: {test_question}\n')
print('Top 3 fragmentos recuperados:')
print('=' * 60)
for i, r in enumerate(test_results, 1):
    print(f'\n[Resultado {i}]')
    print(f'  Página   : {r["metadata"]["page"]}')
    print(f'  Distancia: {r["distance"]:.4f}')
    print(f'  Texto    : {r["text"][:300]}...')

Pregunta: ¿Cuáles son los requisitos socioeconómicos para postular a Beca 18?

Top 3 fragmentos recuperados:

[Resultado 1]
  Página   : desconocida
  Distancia: 0.2076
  Texto    : . En el caso d e los  egresados de educación básica en el 2024, se usarán los promedios de notas de su último y penúltimo año de  estudios. Adicionalmente, se aplican los siguientes filtros: i) Tener una edad menor de 22 años a la fecha de  publicación de las bases, ii) No estar en los padrones de m...

[Resultado 2]
  Página   : desconocida
  Distancia: 0.2137
  Texto    : 34    7. Población Objetivo de Beca 18 y Becas Especiales    La población objetivo de Beca 18 Ordinaria está conformada por los jóvenes egresados  de la educación secundaria con tercio superior 7 y en situación de pobreza o pobreza  extrema de acuerdo con los criterios de focalización establecidos p...

[Resultado 3]
  Página   : desconocida
  Distancia: 0.2146
  Texto    : . Su indicador  corresponde al puntaje obtenido por el  postulan

---
## Paso 6 — Generación con contexto (RAG)

In [24]:
GEN_MODEL = 'gemini-2.5-flash'

SYSTEM_PROMPT = """\
Eres un asistente especializado en la normativa Beca 18 de PRONABEC (Perú).

REGLAS ESTRICTAS:
1. Responde ÚNICAMENTE basándote en los fragmentos del documento que se te proporcionan como contexto.
2. Cuando cites información, SIEMPRE indica el número de página entre paréntesis, por ejemplo: (Página 12).
3. Si el contexto proporcionado NO contiene información suficiente para responder la pregunta,
   responde exactamente: "El documento no contiene información sobre este tema."
4. NO uses conocimiento externo ni hagas suposiciones fuera del contexto dado.
5. Si la pregunta no es sobre Beca 18 o PRONABEC, responde:
   "Esta consulta está fuera del alcance de este sistema. Solo puedo responder preguntas sobre el reglamento Beca 18."
6. Responde en español, de forma clara y estructurada.
"""


def answer_with_context(question: str, k: int = 5) -> dict:
    """
    Recupera contexto y genera respuesta fundamentada con Gemini.
    Retorna dict con: answer, sources.
    """
    # 1. Recuperar fragmentos relevantes
    sources = semantic_search(question, k=k)

    # 2. Construir contexto
    context_parts = []
    for i, src in enumerate(sources, 1):
        page_info = f"Página {src['metadata']['page']}"
        context_parts.append(f'[Fragmento {i} — {page_info}]\n{src["text"]}')
    context_str = '\n\n'.join(context_parts)

    # 3. Prompt al LLM
    user_message = (
        f'CONTEXTO DEL DOCUMENTO:\n{context_str}\n\n'
        f'PREGUNTA DEL USUARIO:\n{question}'
    )

    response = client.models.generate_content(
        model    = GEN_MODEL,
        contents = user_message,
        config   = genai_types.GenerateContentConfig(
            system_instruction = SYSTEM_PROMPT,
            temperature        = 0.1,
            max_output_tokens  = 1024,
        )
    )

    return {
        'answer' : response.text,
        'sources': sources
    }


print('Función answer_with_context definida ✓')

Función answer_with_context definida ✓


In [25]:
# ── Batería de preguntas de prueba ──────────────────────────────────────────

preguntas = [
    # 1. Requisitos de elegibilidad
    '¿Cuáles son los requisitos socioeconómicos para postular a Beca 18?',
    # 2. Modalidades de la beca
    '¿Qué modalidades de Beca 18 existen en la convocatoria 2026?',
    # 3. Monto del estipendio mensual
    '¿Cuál es el monto del estipendio mensual que reciben los becarios?',
    # 4. Obligaciones del estudiante
    '¿Cuáles son las obligaciones del becario durante su formación?',
    # 5. Condiciones para perder la beca
    '¿En qué casos se puede perder o suspender la Beca 18?',
    # 6. Pregunta fuera de tema (anti-alucinación)
    '¿Cuál es la receta para preparar ceviche peruano?',
]

for i, q in enumerate(preguntas, 1):
    print(f'\n{'='*70}')
    print(f'PREGUNTA {i}: {q}')
    print('='*70)
    resultado = answer_with_context(q, k=5)
    print(resultado['answer'])
    print(f'\n  📄 Fuentes: páginas {[s["metadata"]["page"] for s in resultado["sources"]]}')


PREGUNTA 1: ¿Cuáles son los requisitos socioeconómicos para postular a Beca 18?
Los requisitos socioeconómicos para postular a Beca 18 Ordinaria son:

*   Tener condición socioeconómica de pobre no extremo o pobre extremo según el SISFOH (Sistema de Focalización de Hogares) (Fragmento 1, Fragmento 2).
*   Acreditar pobreza o pobreza extrema según el SISFOH es un requisito para Beca 18 Ordinaria (Fragmento 3).
*   La validación de esta condición se realiza mediante consulta en línea a través del Sistema de Focalización de Hogares (SISFOH) del OFIS (Organismo de Focalización e Información Social) (Fragmento 4).

Para las Becas Especiales como VRAEM y Huallaga, en lugar de filtrar por condición socioeconómica, la población se circunscribe a las IIEE (Instituciones Educativas) cuyos ubigeos pertenecen a los distritos comprendidos dentro del ámbito geográfico de la zona del Huallaga o el ámbito de intervención directa y de influencia del VRAEM, según corresponda (Fragmento 1). Sin embargo,

---
## Paso 7 — Interfaz de chat interactiva (ipywidgets)

In [26]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Widgets ──────────────────────────────────────────────────────────────────
style_header = 'style="font-family: Arial, sans-serif; color: #1a237e;"'

title = widgets.HTML(
    value=f'<h2 {style_header}>🎓 Chatbot Beca 18 — PRONABEC 2026</h2>'
          '<p style="color:#555">Consulta el Reglamento Oficial · RDE N.° 033-2026-MINEDU/VMGI-PRONABEC</p>'
)

question_input = widgets.Textarea(
    placeholder='Escribe tu pregunta sobre Beca 18 aquí...',
    layout=widgets.Layout(width='100%', height='80px')
)

k_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description='Chunks (k):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

btn_ask   = widgets.Button(description='🔍 Preguntar',
                           button_style='primary',
                           layout=widgets.Layout(width='140px', height='36px'))
btn_clear = widgets.Button(description='🗑 Borrar',
                           button_style='warning',
                           layout=widgets.Layout(width='120px', height='36px'))

output_answer = widgets.Output()
output_sources = widgets.Accordion(
    children=[widgets.Output()],
    selected_index=None
)
output_sources.set_title(0, '📎 Ver fragmentos fuente recuperados')

status_label = widgets.Label(value='')


# ── Lógica de eventos ─────────────────────────────────────────────────────────
def on_ask_clicked(b):
    question = question_input.value.strip()
    if not question:
        status_label.value = '⚠ Por favor, escribe una pregunta.'
        return

    status_label.value = '⏳ Buscando y generando respuesta...'
    btn_ask.disabled   = True

    with output_answer:
        clear_output(wait=True)

    with output_sources.children[0]:
        clear_output(wait=True)

    try:
        result = answer_with_context(question, k=k_slider.value)

        # Mostrar respuesta
        with output_answer:
            display(HTML(
                f'<div style="background:#e8f5e9;padding:14px;border-radius:8px;'
                f'border-left:4px solid #2e7d32;font-family:Arial,sans-serif;">'
                f'<b style="color:#1b5e20">Respuesta:</b><br><br>'
                f'{result["answer"].replace(chr(10), "<br>")}</div>'
            ))

        # Mostrar fuentes en acordeón
        with output_sources.children[0]:
            for i, src in enumerate(result['sources'], 1):
                page = src['metadata']['page']
                dist = src['distance']
                text_preview = src['text'].replace('<', '&lt;').replace('>', '&gt;')
                display(HTML(
                    f'<div style="background:#f5f5f5;padding:10px;margin:6px 0;'
                    f'border-radius:6px;border-left:3px solid #1565c0;font-size:0.9em;">'
                    f'<b>Fragmento {i}</b> — Página {page} '
                    f'<span style="color:#888">(dist: {dist:.4f})</span><br><br>'
                    f'<span style="font-family:monospace">{text_preview[:500]}...</span></div>'
                ))

        output_sources.set_title(0, f'📎 {len(result["sources"])} fragmentos fuente recuperados')
        status_label.value = f'✅ Respuesta generada · {len(result["sources"])} chunks usados'

    except Exception as e:
        with output_answer:
            display(HTML(f'<p style="color:red">❌ Error: {e}</p>'))
        status_label.value = f'❌ Error: {e}'
    finally:
        btn_ask.disabled = False


def on_clear_clicked(b):
    question_input.value = ''
    with output_answer:
        clear_output()
    with output_sources.children[0]:
        clear_output()
    output_sources.set_title(0, '📎 Ver fragmentos fuente recuperados')
    status_label.value = ''


btn_ask.on_click(on_ask_clicked)
btn_clear.on_click(on_clear_clicked)


# ── Layout final ──────────────────────────────────────────────────────────────
ui = widgets.VBox([
    title,
    widgets.HTML('<b>Tu pregunta:</b>'),
    question_input,
    widgets.HBox([btn_ask, btn_clear, k_slider]),
    status_label,
    widgets.HTML('<hr style="margin:10px 0">'),
    output_answer,
    output_sources,
], layout=widgets.Layout(padding='12px', max_width='860px'))

display(ui)